# lift3d boxes vs Kocur boxes as LSTM features (BrnoCompSpeed)

Same tracks, same calibration, same LSTM, same fine-tuning: only the **3D-box step** changes.

- **lift3d arm**: the existing prepared runs in `runs/brno/<recording>` (the inputs of the 1.18 km/h result, `exp_024_ens_bs64`).
- **Kocur arm**: the authors' YOLOv6-3D (Macko, Gajdošech & Kocur 2025; Small model, 640×360, the paper's best median)
  runs on the same frames with the same calibration (`system_dubska_optimal_calib.json`). Each YOLO + ByteTrack box takes the
  Kocur box that overlaps it, which becomes `[cx, cy, cz, length, width, height]` on the road plane. Like lift3d, size is fixed
  per track (median) and centers are smoothed the same way.
- Both arms fine-tune the I-24 checkpoint with the 1.18 setup (train sessions 1-3 center, 30 epochs, lr 1e-3, batch 64,
  seeds 0-2, ensemble) and are scored once on sessions 4-6 center. No settings are tuned on the test set.

Note: the Kocur weights were trained on BrnoCompSpeed sessions 0-3 (split C), so its boxes on our training sessions are
in-sample, while on test sessions 4-6 they are not, as in the authors' own evaluation.

Use a GPU runtime (T4). Everything is saved to Drive as it finishes, so a rerun skips finished steps.
Outputs: `runs/kocur_brno/` (raw Kocur boxes), `runs/brno_kocur/` (prepared Kocur-arm runs), `runs/kocur_compare/` (fine-tuning + results).

In [ ]:
# ---- settings ----
TRAIN = ['session1_center', 'session2_center', 'session3_center']
TEST = ['session4_center', 'session5_center', 'session6_center']
SEEDS = [0, 1, 2]
KOCUR_WEIGHTS = 'checkpoints/small/yolov6_small_3d_transform.pt'
KOCUR_SIZE = ['360', '640']   # model input h w
BRANCH = 'claude/project-thread-rwrrf5'

In [ ]:
import os, json
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

PROJECT = next((c for c in ['/content/drive/MyDrive/Final-Project-CHULA',
                            '/content/drive/Shareddrives/Final-Project-CHULA'] if os.path.isdir(c)), None)
assert PROJECT, 'Add a shortcut to Final-Project-CHULA in MyDrive, or set PROJECT manually'
BRNO = f'{PROJECT}/Brno/data/2016-ITS-BrnoCompSpeed'
LIFT_RUNS = f'{PROJECT}/runs/brno'          # lift3d arm (existing)
KOCUR_RAW = f'{PROJECT}/runs/kocur_brno'    # raw Kocur boxes
KOCUR_RUNS = f'{PROJECT}/runs/brno_kocur'   # Kocur arm, prepared like runs/brno
CMP = f'{PROJECT}/runs/kocur_compare'       # fine-tuning runs and results
BASE = f'{PROJECT}/runs/v2/3d/best.pt'
assert os.path.isfile(BASE), BASE

for rec in TRAIN + TEST:
    run = f'{LIFT_RUNS}/{rec}'
    need = ['detections.csv', 'run_config.json', 'windows.json', 'car_eval.csv', 'summary.json']
    need += ['labeled_windows.json'] if rec in TRAIN else []
    missing = [n for n in need if not os.path.isfile(f'{run}/{n}')]
    calib = json.load(open(f'{run}/summary.json')).get('calibration') if not missing else None
    print(f'{rec}: lift3d run {"OK" if not missing else "MISSING " + str(missing)}, calibration {calib}')
    assert os.path.isfile(f'{BRNO}/dataset/{rec}/video.avi'), f'no video for {rec}'

In [ ]:
%cd /content
!rm -rf capcapcar-speed kocur
!git clone -q -b {BRANCH} https://github.com/WalkerCze3/capcapcar-speed.git
!git clone -q --depth 1 https://github.com/gajdosech2/Vehicle-Speed-Estimation-YOLOv6-3D.git kocur
!pip install -q addict pycocotools
%cd /content/capcapcar-speed

## Step 1: Kocur boxes on each recording, prepared like the lift3d runs

Per recording: run Kocur on the same 10 minutes / frames as the YOLO run, turn its boxes into the model's 3D input on our
tracks, then run the same Brno preparation as the lift3d arm (windows, official matching, labeled windows).
Roughly 3-8 minutes per recording on a T4.

In [ ]:
import shutil, subprocess

def sh(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    for line in p.stdout:
        if 'Warning' not in line and 'warn(' not in line:
            print(line, end='', flush=True)
    if p.wait():
        raise subprocess.CalledProcessError(p.returncode, cmd)

for rec in TRAIN + TEST:
    if os.path.isfile(f'{KOCUR_RUNS}/{rec}/summary.json'):
        print(f'===== {rec}: done earlier, skipping'); continue
    print(f'===== {rec} =====')
    lift_cfg = json.load(open(f'{LIFT_RUNS}/{rec}/run_config.json'))
    session = f'{BRNO}/dataset/{rec}'
    calib = f'{BRNO}/results/{rec}/system_dubska_optimal_calib.json'
    raw, prep = f'runs/kocur_brno/{rec}', f'runs/brno_kocur/{rec}'
    if not os.path.isfile(f'{KOCUR_RAW}/{rec}/detections_kocur.csv'):
        sh(['python', 'baselines/kocur_brno/run_kocur_brno.py', '--kocur-repo', '/content/kocur',
            '--weights', KOCUR_WEIGHTS, '--yolo-size', *KOCUR_SIZE, '--session-dir', session, '--calib', calib,
            '--max-seconds', str(lift_cfg['max_seconds']), '--out', raw])
        shutil.copytree(raw, f'{KOCUR_RAW}/{rec}', dirs_exist_ok=True)
    sh(['python', 'baselines/kocur_brno/kocur_boxes3d.py', '--run-dir', f'{LIFT_RUNS}/{rec}',
        '--kocur-dir', f'{KOCUR_RAW}/{rec}', '--session-dir', session, '--calib', calib, '--out', f'{prep}/kocur_boxes3d.csv'])
    sh(['python', 'scripts/brno_eval_cli.py', '--session-dir', session, '--calib', calib, '--checkpoint', BASE,
        '--detections', f'{LIFT_RUNS}/{rec}/detections.csv', '--boxes3d', f'{prep}/kocur_boxes3d.csv',
        '--max-seconds', str(lift_cfg['max_seconds']), '--smooth', str(lift_cfg['smooth']),
        '--stride', str(lift_cfg['stride']), '--out-dir', prep])
    shutil.copytree(prep, f'{KOCUR_RUNS}/{rec}', dirs_exist_ok=True)
    print('saved', f'{KOCUR_RUNS}/{rec}')

## Step 2: fine-tune both arms with the 1.18 setup (3 seeds each) and ensemble

In [ ]:
FT = {'epochs': 30, 'lr': 1e-3, 'freeze': 'none', 'renorm': 'keep', 'loss': 'mse', 'batch_size': 64}
ARMS = {'lift3d': LIFT_RUNS, 'kocur': KOCUR_RUNS}

def experiment(cfg):
    out = f'{CMP}/runs/experiments/{cfg["name"]}'
    if os.path.isfile(f'{out}/results.json'):
        print(f'== {cfg["name"]}: done earlier'); return
    print(f'== {cfg["name"]}')
    sh(['python', 'scripts/brno_experiment.py', '--project', CMP, '--out-dir', out, '--config', json.dumps(cfg)])

for arm, runs_root in ARMS.items():
    names = []
    for seed in SEEDS:
        name = f'{arm}_s{seed}'
        experiment({'name': name, 'base_checkpoint': BASE, 'runs_root': runs_root, 'train': TRAIN, 'test': TEST,
                    'finetune': {**FT, 'seed': seed}})
        names.append(name)
    experiment({'name': f'{arm}_ens', 'runs_root': runs_root, 'test': TEST, 'finetune': None, 'ensemble_of': names})

## Results

In [ ]:
import pandas as pd
rows, cars = [], {}
for arm in ARMS:
    for name in [f'{arm}_s{s}' for s in SEEDS] + [f'{arm}_ens']:
        r = json.load(open(f'{CMP}/runs/experiments/{name}/results.json'))
        m, g = r['test_model'], r['test_geometry_median']
        rows.append({'run': name, 'cars': m.get('n'), 'LSTM MAE km/h': m.get('mae'), 'LSTM median': m.get('median'),
                     'LSTM bias': m.get('bias'), 'geometry MAE km/h': g.get('mae'),
                     'train windows': r.get('n_train_windows')})
    cars[arm] = pd.read_csv(f'{CMP}/runs/experiments/{arm}_ens/test_cars.csv')
table = pd.DataFrame(rows)

# Same cars in both arms (each arm can match a slightly different set of ground-truth cars)
key = ['recording', 'gt_id']
both = cars['lift3d'].merge(cars['kocur'], on=key, suffixes=('_lift3d', '_kocur'))
both = both[both['model_kmh_lift3d'].notna() & both['model_kmh_kocur'].notna()]
common = {arm: {'common cars': len(both),
                'LSTM MAE km/h': (both[f'model_kmh_{arm}'] - both['gt_kmh_lift3d']).abs().mean(),
                'geometry MAE km/h': (both[f'median_kmh_{arm}'] - both['gt_kmh_lift3d']).abs().mean()} for arm in ARMS}
common = pd.DataFrame(common).T
per_rec = pd.DataFrame({arm: c.assign(err=(c['model_kmh'] - c['gt_kmh']).abs()).groupby('recording')['err'].mean()
                        for arm, c in cars.items()})

table.to_csv(f'{CMP}/summary_runs.csv', index=False)
common.to_csv(f'{CMP}/summary_common_cars.csv')
per_rec.to_csv(f'{CMP}/summary_per_recording.csv')
display(table.round(3)); display(common.round(3)); display(per_rec.round(3))

In [ ]:
# Box sizes each arm feeds the model (median per track), and how many tracked boxes got a Kocur box
for arm, root in ARMS.items():
    sizes = pd.concat([pd.read_csv(f'{root}/{rec}/boxes3d.csv').groupby('track_id')[['length', 'width', 'height']].median()
                       for rec in TRAIN + TEST])
    print(arm, 'median track size L/W/H (m):', sizes.median().round(2).tolist(), f'({len(sizes)} tracks)')